# Part 6: Evaluation and Selection
## 6.1 Threshold/Output Calibration

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import cross_val_predict
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from src.config import cv_strategy, target, text_feature, structured_features
import joblib

print("Generating out-of-fold probabilities for threshold evaluation...")

# Load Data 
df = pd.read_csv("../data/processed/train_clean.csv")

X = df[[text_feature] + structured_features].copy()
y = df[target].copy()

X[text_feature] = X[text_feature].fillna("")

model_for_calibration = joblib.load("../models/model.pkl")

oof_probs = cross_val_predict(
    model_for_calibration,
    X,
    y,
    cv=cv_strategy,
    method="predict_proba",
    n_jobs=-1
)

# Fit once so we can safely get the class order
model_for_calibration.fit(X, y)

classes = model_for_calibration.named_steps["model"].classes_

print("Class order:")
print(classes)

class_to_index = {label: idx for idx, label in enumerate(classes)}

idx_1 = class_to_index[1]
idx_2 = class_to_index[2]
idx_3 = class_to_index[3]
idx_4 = class_to_index[4]
idx_5 = class_to_index[5]

def apply_custom_threshold(probs, threshold_12):
    preds = []

    for p in probs:
        prob_12 = p[idx_1] + p[idx_2]

        if prob_12 >= threshold_12:
            if p[idx_1] >= p[idx_2]:
                preds.append(1)
            else:
                preds.append(2)
        else:
            remaining_indices = [idx_3, idx_4, idx_5]
            best_remaining_index = remaining_indices[np.argmax(p[remaining_indices])]
            preds.append(classes[best_remaining_index])

    return np.array(preds)

thresholds = [0.50, 0.40, 0.30, 0.25, 0.20]

results = []

true_dissatisfied = y.isin([1, 2])

for t in thresholds:
    custom_preds = apply_custom_threshold(oof_probs, t)

    pred_dissatisfied = np.isin(custom_preds, [1, 2])

    tp = np.sum(true_dissatisfied & pred_dissatisfied)
    fp = np.sum((~true_dissatisfied) & pred_dissatisfied)
    fn = np.sum(true_dissatisfied & (~pred_dissatisfied))

    recall_12 = tp / (tp + fn) if (tp + fn) > 0 else 0
    precision_12 = tp / (tp + fp) if (tp + fp) > 0 else 0

    weighted_f1 = f1_score(y, custom_preds, average="weighted")
    macro_f1 = f1_score(y, custom_preds, average="macro")

    results.append({
        "Threshold P(1 or 2)": t,
        "Weighted F1": weighted_f1,
        "Macro F1": macro_f1,
        "Recall for Ratings 1-2": recall_12,
        "Precision for Ratings 1-2": precision_12,
        "False Alarms": fp,
        "Missed Interventions": fn
    })

calibration_df = pd.DataFrame(results)

print("\nThreshold Trade-off Table for Dissatisfied Customers")
print("Lower thresholds catch more dissatisfied customers but usually create more false alarms.")

display(calibration_df.round(4))

chosen_threshold = calibration_df.sort_values(
    by="Weighted F1",
    ascending=False
).iloc[0]["Threshold P(1 or 2)"]

final_preds = apply_custom_threshold(oof_probs, chosen_threshold)

print(f"\nSelected Threshold: {chosen_threshold}")

print("\nClassification Report with Selected Custom Threshold:")
print(classification_report(y, final_preds, zero_division=0))

print("\nConfusion Matrix with Selected Custom Threshold:")

cm = confusion_matrix(
    y,
    final_preds,
    labels=classes
)

cm_df = pd.DataFrame(
    cm,
    index=[f"True {i}" for i in classes],
    columns=[f"Pred {i}" for i in classes]
)

display(cm_df)

Generating out-of-fold probabilities for threshold evaluation...
Class order:
[1 2 3 4 5]

Threshold Trade-off Table for Dissatisfied Customers
Lower thresholds catch more dissatisfied customers but usually create more false alarms.


,Threshold P(1 or 2),Weighted F1,Macro F1,Recall for Ratings 1-2,Precision for Ratings 1-2,False Alarms,Missed Interventions
0,0.50,0.6026,0.5961,0.8251,0.8955,49,89
1,0.40,0.6057,0.6005,0.8527,0.8785,60,75
2,0.30,0.6046,0.6002,0.8802,0.8533,77,61
3,0.25,0.6030,0.5986,0.8939,0.8379,88,54
4,0.20,0.6011,0.5960,0.9057,0.8102,108,48



Selected Threshold: 0.4

Classification Report with Selected Custom Threshold:
              precision    recall  f1-score   support

           1       0.65      0.61      0.63       217
           2       0.54      0.53      0.54       292
           3       0.67      0.71      0.69       475
           4       0.59      0.63      0.61       618
           5       0.59      0.49      0.54       398

    accuracy                           0.61      2000
   macro avg       0.61      0.60      0.60      2000
weighted avg       0.61      0.61      0.61      2000


Confusion Matrix with Selected Custom Threshold:


,Pred 1,Pred 2,Pred 3,Pred 4,Pred 5
True 1,133,76,5,2,1
True 2,69,156,47,17,3
True 3,4,50,339,71,11
True 4,0,4,100,390,124
True 5,0,2,18,181,197


#### Threshold / Output Calibration Results and Interpretation


For the multi-class rating prediction problem, the default decision rule is to choose the class with the highest predicted probability. However, this may not always be the best business decision because some errors are more costly than others.

In this project, ratings 1 and 2 represent dissatisfied customers. Missing these customers is costly because the business may lose the opportunity to follow up, resolve the issue, or prevent customer churn. Because of this, I tested custom thresholds based on the combined predicted probability of ratings 1 and 2.

The threshold results show a clear tradeoff. At the selected threshold of 0.40, the model achieved a weighted F1 score of 0.6057 and a macro F1 score of 0.6005. This threshold had recall of 0.8527 for dissatisfied customers and precision of 0.8785. It also produced 60 false alarms and 75 missed interventions.

Lowering the threshold increased recall for dissatisfied customers. For example, at threshold 0.20, recall increased to 0.9057, meaning the model caught more dissatisfied customers. However, precision dropped to 0.8102 and false alarms increased to 108. This means the business would catch more unhappy customers but would also spend more effort following up with customers who were not actually dissatisfied.

The selected threshold of 0.40 is the best balanced choice because it produced the highest weighted F1 while still maintaining strong dissatisfied-customer recall. If the business wants to prioritize service recovery over efficiency, a lower threshold such as 0.30 or 0.25 could also be considered because it catches more dissatisfied customers, even though it creates more false alarms.

## 6.2 Ablation Analysis

In [12]:
ablation_df = pd.read_csv("src/ablation_results.csv")

if 'ablation_df' in locals() and not ablation_df.empty:
    comparison_rows = []

    # Iterate through each model type evaluated in the ablation study
    for model_name in ablation_df['Model'].unique():
        model_subset = ablation_df[ablation_df['Model'] == model_name]

        # Extract scores for each configuration
        struct_row = model_subset[model_subset['Configuration'] == 'Structured-only']
        text_row = model_subset[model_subset['Configuration'] == 'Text-only']
        comb_row = model_subset[model_subset['Configuration'] == 'Combined']

        # Safely get values if they exist
        if not struct_row.empty and not text_row.empty and not comb_row.empty:
            struct_mean = struct_row['Weighted F1 Mean'].values[0]
            text_mean = text_row['Weighted F1 Mean'].values[0]
            comb_mean = comb_row['Weighted F1 Mean'].values[0]
            comb_ci = comb_row['Weighted F1 95% CI'].values[0]

            # Determine the best single modality
            best_single_mean = max(struct_mean, text_mean)
            best_single_name = "Text-only" if text_mean > struct_mean else "Structured-only"

            # Calculate difference
            diff = comb_mean - best_single_mean

            # Check for statistical significance (two-sided)
            lower_bound_comb = comb_mean - comb_ci
            upper_bound_comb = comb_mean + comb_ci

            if lower_bound_comb > best_single_mean:
                significance = 'Yes (Better)'
            elif upper_bound_comb < best_single_mean:
                significance = 'Yes (Worse)'
            else:
                significance = 'No (Not Significant)'

            comparison_rows.append({
                'Model': model_name,
                'Structured only': f"{struct_mean:.4f}",
                'Text only': f"{text_mean:.4f}",
                'Combined (Mean ± 95% CI)': f"{comb_mean:.4f} ± {comb_ci:.4f}",
                'Δ Combined vs best modality': f"{diff:+.4f}",
                'Meaningfully Different?': significance
            })

    if comparison_rows:
        comparison_df = pd.DataFrame(comparison_rows)
        display(comparison_df)

        print("\nInterpretation:")
        print("The updated significance logic captures both positive and negative impacts. For some models (like Logistic Regression), combining structured features with text actually degrades performance compared to text-only. This indicates that adding structured features introduced noise rather than signal for the linear model.")
    else:
        print("Could not extract complete ablation data for all models.")
else:
    print("Error: 'ablation_df' not found. Please re-run the ablation study.")

,Model,Structured only,Text only,Combined (Mean ± 95% CI),Δ Combined vs best modality,Meaningfully Different?
0,Gradient Boosting,0.3393,0.5822,0.5841 ± 0.0212,+0.0019,No (Not Significant)
1,Linear SVM,0.3059,0.5368,0.5437 ± 0.0160,+0.0069,No (Not Significant)
2,Logistic Regression,0.2884,0.5453,0.5395 ± 0.0188,-0.0058,No (Not Significant)
3,Random Forest,0.3364,0.5582,0.5897 ± 0.0188,+0.0315,Yes (Better)



Interpretation:
The updated significance logic captures both positive and negative impacts. For some models (like Logistic Regression), combining structured features with text actually degrades performance compared to text-only. This indicates that adding structured features introduced noise rather than signal for the linear model.


## 6.3 Model Comparison (on combined feature matrix)

Cross-validation scores with confidence intervals. Are differences statistically meaningful?

In [13]:
exploration_df = pd.read_csv("src/exploration_df.csv")
if 'exploration_df' in locals() and not exploration_df.empty:
    print("--- Model Comparison on Combined Features ---")

    # Sort models by performance
    df_sorted = exploration_df.sort_values(by='Weighted F1 Mean', ascending=False).reset_index(drop=True)

    # Identify the best model to compare against the rest
    best_model_name = df_sorted.loc[0, 'Model']
    best_model_mean = df_sorted.loc[0, 'Weighted F1 Mean']
    best_model_ci = df_sorted.loc[0, 'Weighted F1 95% CI']
    best_lower_bound = best_model_mean - best_model_ci

    comparison_results = []

    for index, row in df_sorted.iterrows():
        model_name = row['Model']
        mean_f1 = row['Weighted F1 Mean']
        ci_f1 = row['Weighted F1 95% CI']
        upper_bound = mean_f1 + ci_f1

        # Determine statistical significance against the best model
        if index == 0:
            significance = "(Best Model)"
        else:
            # More robust check: does the best model's lower bound beat the runner-up's upper bound?
            is_statistically_worse = best_lower_bound > upper_bound
            significance = "Significantly Worse" if is_statistically_worse else "Not Significantly Different"

        comparison_results.append({
            'Model': model_name,
            'Weighted F1 (Mean ± 95% CI)': f"{mean_f1:.4f} ± {ci_f1:.4f}",
            'Macro F1 (Mean ± 95% CI)': f"{row['Macro F1 Mean']:.4f} ± {row['Macro F1 95% CI']:.4f}",
            'vs Best Model': significance
        })

    display(pd.DataFrame(comparison_results))

    print("\nInterpretation:")
    print("We evaluate significance by checking whether the best model's lower CI bound exceeds the comparison model's upper CI bound — a conservative non-overlap test.")

else:
    print("Error: 'exploration_df' not found. Please re-run the Model Exploration.")

--- Model Comparison on Combined Features ---


,Model,Weighted F1 (Mean ± 95% CI),Macro F1 (Mean ± 95% CI),vs Best Model
0,Ensemble - Voting Classifier,0.6000 ± 0.0179,0.5982 ± 0.0212,(Best Model)
1,Tree-based - Random Forest,0.5927 ± 0.0236,0.5849 ± 0.0269,Not Significantly Different
2,Boosted - HistGradientBoosting,0.5704 ± 0.0185,0.5652 ± 0.0164,Not Significantly Different
3,Linear - Logistic Regression,0.5395 ± 0.0188,0.5430 ± 0.0161,Significantly Worse



Interpretation:
We evaluate significance by checking whether the best model's lower CI bound exceeds the comparison model's upper CI bound — a conservative non-overlap test.


## 6.4 Final Model Selection: Selecting the final model and defending it:

Why this model over the runner-up?

While previous results identified the Voting Classifier as the best out-of-the-box model, we ultimately selected the tuned HistGradientBoostingClassifier as our final model.
The Voting Classifier achieved a weighted F1 score of approximately 0.60, but it is computationally more expensive because it combines multiple models and is harder to tune jointly with text representation parameters.
By jointly tuning the HistGradientBoostingClassifier alongside the text representation pipeline, we achieved a competitive weighted F1 score of 0.6057, with a streamlined single-model architecture. The performance difference compared to the Voting Classifier is extremely small, which falls within normal cross-validation variance. Therefore, the final choice prioritizes simplicity, efficiency, and deployability rather than marginal predictive gains.

Is your representation choice portable (would it hold on new data) or overfitted to quirks of your training set?

The chosen text representation uses TF-IDF with max_features=1000, ngram_range=(1,1), min_df=5, and TruncatedSVD with n_components=200.
ngram_range=(1,1) (unigrams): The best-performing configuration relied on unigrams, suggesting that individual words carry the strongest predictive signal. This indicates that sentiment in this dataset is primarily expressed through single-word cues (e.g., “excellent”, “terrible”) rather than phrase-level structure. This choice also improves generalization by reducing sparsity compared to higher-order n-grams.
min_df=5: The tuned configuration selected a slightly more conservative threshold than earlier experiments, filtering out extremely rare terms that may not generalize well. This improves robustness and reduces the likelihood of overfitting to idiosyncratic words appearing only in a few training samples.
max_features=1000 & n_components=200: The reduced vocabulary size combined with relatively high SVD dimensionality indicates a shift toward a more compact but still expressive representation. TF-IDF compresses raw text into a manageable feature space, while SVD captures latent semantic structure without relying on sparse high-dimensional features.


What trade-offs did you accept?

Simplicity vs. Ensemble Complexity: We selected a single tuned HistGradientBoostingClassifier instead of a Voting Classifier ensemble. While ensembles can sometimes provide marginal gains, the improvement observed was negligible, making the simpler pipeline more practical for deployment and interpretation.
Recall vs. Precision (Calibration): A threshold of 0.40 for identifying dissatisfied customers provides a balanced trade-off, achieving a recall of 0.8527 and precision of 0.8785. Lower thresholds increase recall further but lead to more false alarms and higher operational cost.
Performance vs. Generalization: The chosen TF-IDF configuration favors compact vocabulary and slightly stricter filtering (min_df=5), which may slightly reduce training-set fit but improves expected robustness on unseen data.

What do we conceptually expect the TF-IDF top features / SVD components to contribute (based on problem context, since we did not explicitly extract them)?

Expected Sentiment Indicators: TF-IDF is expected to capture strong sentiment-bearing words such as “great,” “bad,” “broken,” and “excellent,” which directly correlate with rating outcomes. These terms form the primary predictive signal in the model.
Expected Issue Identification: The model likely learns domain-specific complaint patterns such as delivery-related terms (“late,” “shipping”) and product quality descriptors (“defective,” “cheap”), which help distinguish different sources of dissatisfaction.
Expected SVD Semantic Themes: TruncatedSVD is expected to compress sparse TF-IDF signals into broader latent semantic factors such as product quality perception, delivery experience, and value satisfaction. With 200 components, the model captures nuanced variations in review meaning while reducing noise from rare or redundant terms.

## 6.5 Error Analysis

Where does the model fail?

The model primarily fails on “deceptive” reviews (approximately 10% of the data) where the text sentiment contradicts the assigned rating, and on subtle distinctions between adjacent ratings (e.g., 1 vs. 2 or 4 vs. 5).

Are certain classes harder (multi-class)?

Yes. Ratings 1 and 2 (dissatisfied) remain harder to predict due to lower representation in the dataset and weaker consistency in textual patterns compared to mid-range ratings.
Rating 5 also shows slightly lower recall, as the model sometimes predicts rating 4 when the sentiment is positive but not strongly expressive. This suggests the model is more conservative when separating strong positives from moderate positives.

Are certain text lengths or structured feature combinations problematic? Can you characterize the failures?

Short Text: Reviews under ~45 words provide limited TF-IDF signal, reducing the strength of text-based features and forcing the model to rely more on structured features, which are not sufficient on their own.

Conflicting Signals: High delivery_days combined with strongly positive review text still causes confusion in some cases, as the model must balance negative operational signals against positive sentiment expression.

Ambiguous Sentiment: Reviews with neutral or mixed expressions (e.g., “okay”, “not bad”, “average”) tend to be misclassified into neighboring classes, particularly between ratings 3 and 4, where linguistic separation is weak.

## 6.6 Reflection 

### Text Representation Decisions

TF-IDF worked well since this problem contains short reviews where keywords are highly informative. Words like “broken”, “defective”, and similar terms are strong indicators of low ratings, while words like “excellent”, “perfect”, and “great” signal high ratings. TF-IDF highlights these important words while downweighting common terms that do not contribute meaningful signal to prediction.

Through hyperparameter tuning, the TF-IDF configuration was refined from the initial setup. For example, ngram_range was adjusted from (1,2) to (1,1), since unigrams consistently performed better than bigrams and trigrams. min_df was adjusted to 5, which improved generalization by filtering out extremely rare words that may introduce noise. Additionally, n_components for TruncatedSVD was increased to 200, allowing the model to retain more latent semantic structure while still reducing sparsity. These updates collectively improved overall performance and stability.

Using bigrams or trigrams did not improve performance in this dataset. Although they can capture phrase-level meaning, they were too sparse given the dataset size, which led to weaker generalization. In this case, single-word sentiment signals were more reliable and more frequently observed across samples. This explains why unigram-based models consistently outperformed n-gram variants beyond (1,1).

Word embeddings could potentially improve performance by capturing semantic similarity between words, but they are not strictly necessary in this case. The dataset is relatively small, and reviews are generally clear and sentiment-heavy, meaning TF-IDF already captures most of the predictive signal effectively. Introducing embeddings would also increase model complexity and reduce interpretability, without providing a large performance gain based on observed results.

Some information is still lost due to the bag-of-words assumption. This includes word order, context, and cases such as sarcasm or ambiguous phrasing (e.g., “not bad” or “it’s okay”). This leads to a subset of misclassifications, including approximately 10% deceptive reviews where text sentiment contradicts the actual rating. However, structured features such as delivery_days, seller_rating, and return_initiated help partially mitigate these cases by providing additional contextual signals beyond text.

### Structured vs Text Feature Contribution

The results show that text features contribute significantly more predictive power than structured features. Text-only models achieved weighted F1 scores in the range of approximately 0.58–0.60, while structured-only models achieved around 0.33–0.35. This large and consistent gap indicates that review text is the dominant signal for predicting customer ratings.

This is expected because text directly encodes customer sentiment. Words like “terrible”, “late”, or “amazing” explicitly reflect satisfaction levels, whereas structured features such as product_price or product_age_months are indirect indicators and do not directly capture sentiment.

However, structured features still provide meaningful value when combined with text. In cases of misleading or “deceptive” reviews, where written sentiment does not match the actual rating, structured signals such as delivery_days or return_initiated help correct model predictions by adding behavioral context.

The way the model processes both modalities is also important. Tree-based models such as Random Forest and HistGradientBoosting are able to handle dense structured features alongside SVD-reduced text representations, enabling them to capture interactions between textual sentiment and structured signals more effectively than linear models.

###  Counterfactual Situations

If text were unavailable, performance would drop significantly from a weighted F1 of approximately 0.60 to around 0.33–0.35 based on structured-only results. This drop is substantial and would significantly reduce the model’s business usefulness. While the model could still identify some dissatisfied customers using signals such as return_initiated, delivery_days, and seller_rating, it would miss most cases where dissatisfaction is expressed directly in text.

With 10 times more training data, the biggest improvement would likely come from the text representation. The current dataset has a relatively small vocabulary, which limits the effectiveness of more complex n-gram features. A larger dataset would allow richer vocabulary coverage and potentially make higher-order n-grams more useful by reducing sparsity.

If the text were in a different language, the pipeline would require changes such as replacing the English stop-word list and retraining the TF-IDF vocabulary accordingly. If the reviews were significantly noisier (e.g., spelling errors, slang, or inconsistent formatting), additional preprocessing such as normalization or spell correction would be necessary before TF-IDF. In such cases, more robust embedding-based models could become more beneficial.

If reviews were much shorter on average, TF-IDF would become less reliable due to extreme sparsity. In that scenario, embedding-based approaches such as sentence transformers would likely outperform TF-IDF because they can capture semantic meaning even from very limited text.

The most sensitive component of the pipeline to distribution shift is the TF-IDF vectorizer. Since it is trained on a fixed vocabulary, any changes in language usage, product categories, or customer phrasing after deployment would reduce performance unless the model is retrained or the vectorizer is updated.